# C4-classical-ml-practice — Review

Work through this notebook *after* the three lesson sessions and
(ideally) the practice sets.
It is a consolidation tool: a concept summary, the pandas and
pipeline idiom sheets, a self-quiz spanning every taught concept, and
pointers to what to redo.
Quiz answers are collapsed at the very end — commit to your answers
before looking.

In [ ]:
import numpy as np
import pandas as pd

## Concept summary

| Concept | One-line summary | Key fact to retain | Session |
|---|---|---|---|
| `pandas-basics` | DataFrame = named columns (Series), each with its own dtype; masks, `.loc`/`.iloc`, `groupby` | parenthesize every comparison in a combined mask; `df["c"]` is 1-D, `df[["c"]]` is 2-D | 1 |
| `csv-data-loading` | `pd.read_csv` / `to_csv(index=False)`; first-look ritual: `shape`, `head`, `dtypes`, `describe`, `value_counts` | a numeric column loading as `object` means a non-numeric cell is hiding in it | 1 |
| (the bridge) | explicit `FEATURES` list → `df[FEATURES].to_numpy()`, label column separate | `X` must be float `(n, d)`; `y` may be text `(n,)`; column order = list order | 1 |
| `knn` | distances to all training rows → `argsort` → majority vote of the $k$ nearest | no fitting — kNN memorizes; $k=1$ train accuracy is 1.0 *by construction* (overfitting); odd $k$ dodges binary ties | 2 |
| `feature-scaling` | feature $j$ contributes $\sim s_j^2$ to squared distance — units pick the winner unless you standardize: $z = (x-\mu)/\sigma$ | $\mu, \sigma$ from **training rows only**, applied to train, test, and every query alike | 2 |
| `sklearn-pipelines` | `Pipeline([("scaler", ...), ("knn", ...)])` welds preprocessing + model into one estimator | `fit` fits the scaler on the given (training) data only; `predict`/`score` reuse the stored statistics — the two leakage traps become unwritable | 3 |
| `cross-validation` | $K$ folds → $K$ models, each validated on its held-out fold; report mean ± spread | each row validated exactly once, trained on $K{-}1$ times; CV compares candidates on *training* data — the test set still waits, and is touched once | 3 |

## Idiom sheet

**pandas (Session 1):**

```python
df = pd.read_csv("data/beans.csv")              # load
df.shape; df.head(); df.dtypes; df.describe()   # first-look ritual
df["species"].value_counts()                    # class balance
df[(df["mass_g"] > 0.7) & (df["species"] == "cava")]   # mask (parentheses!)
mask.sum(); mask.mean()                         # count / fraction
df.loc[mask, ["length_mm", "mass_g"]]           # rows by mask, columns by name
df.iloc[0]                                      # row by position
df.groupby("species")["mass_g"].mean()          # per-class summary
```

**The bridge (every applied task starts here):**

```python
FEATURES = ["length_mm", "width_mm", "mass_g", "moisture_pct"]
X = df[FEATURES].to_numpy()          # (n, d) float64  -- verify!
y = df["species"].to_numpy()         # (n,)
```

**First-principles kNN (Session 2 — the sklearn-banned register):**

```python
d2 = ((X_tr[None, :, :] - X_q[:, None, :]) ** 2).sum(axis=2)   # (m, n)
nearest = np.argsort(d2, axis=1)[:, :k]                        # (m, k)
preds = (y_tr[nearest].mean(axis=1) > 0.5).astype(int)         # 0/1 vote, odd k
```

**Manual standardization (train statistics, always):**

```python
mu, sd = X_tr.mean(axis=0), X_tr.std(axis=0)
Z_tr, Z_te = (X_tr - mu) / sd, (X_te - mu) / sd
```

**The sanctioned sklearn surface (Sessions 2–3):**

```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=SEED, stratify=y)
pipe = Pipeline([("scaler", StandardScaler()),
                 ("knn", KNeighborsClassifier(n_neighbors=5))])
pipe.fit(X_tr, y_tr); pipe.score(X_te, y_te)      # leak-proof by construction
scores = cross_val_score(pipe, X_tr, y_tr, cv=5)  # pipeline INSIDE the CV
scores.mean(), scores.std()                       # report mean, never max
```

**The honest choose-k arc (Session 3 §5):**
CV-score each candidate on the training part → best mean, smallest
$k$ on ties → refit on the full training part → one test evaluation.

## Self-quiz

Work every item on paper (and NumPy where asked) *before* opening the
answers at the end.

1. `df["mass_g"]` versus `df[["mass_g"]]`: what does each return, and
   what are the shapes after `.to_numpy()` for a 90-row table?
2. Fix this line and say why it fails:
   `df[df["mass_g"] > 0.7 & (df["species"] == "cava")]`.
3. What does `index=False` in `to_csv` prevent — and, on loading,
   what does a numeric-looking column with dtype `object` tell you?
4. Write the three-line bridge from a DataFrame with columns
   `["status", "temp_c", "vibration_mm_s", "pressure_pa"]` (label:
   `status`) to `(X, y)` using `temp_c` and `vibration_mm_s` only.
5. Training rows $(0,0), (2,0), (0,3), (4,4)$ with labels
   $0, 0, 1, 1$; query $(1, 1)$.
   Compute all four squared distances and classify with $k=1$ and
   $k=3$.
6. Why is $k=1$ training accuracy always 1.0 (distinct rows), and
   which C1 concept does that illustrate?
7. 7 queries, 50 training rows, 4 features: give the shapes of the
   difference block `X_tr[None,:,:] - X_q[:,None,:]`, the distance
   matrix, and the prediction array.
8. A column has $\mu = 10$, $\sigma = 2$.
   Standardize $x = 14$; then state the mean and variance of the
   whole standardized column, and the F5 facts that guarantee them.
9. Features with spreads $\approx 0.1$ and $\approx 100$: what is the
   approximate influence ratio in squared Euclidean distance, and
   which feature picks the neighbors?
10. Give both reasons the scaler's $\mu, \sigma$ must come from
    training rows only.
11. List, in order, what happens inside
    `Pipeline([("scaler", ...), ("knn", ...)]).fit(X_tr, y_tr)` —
    and what happens to `X_te` during `.score(X_te, y_te)`.
12. Why does passing the *pipeline* to `cross_val_score` prevent the
    pre-scaling leak?
    What exactly leaks in the pre-scaled protocol?
13. 5-fold CV on 40 rows: how many models, trained on how many rows
    each, each row validated how often?
    And: when is the mean of fold accuracies exactly the pooled
    accuracy?
14. Rank as "the number to report": best fold score, mean CV score of
    the chosen model, final once-only test accuracy — and justify.
15. Exam craft (paraphrased rule): on the applied problem, which
    model family does the paper's sklearn permission actually cover,
    and which five imports from this unit form the sanctioned
    toolkit?

## What to redo, per weak spot

| Shaky on … | Redo |
|---|---|
| brackets, masks, `groupby` | Session 1 §3–5; `p01`, `p20` |
| CSV loading + first look | Session 1 §2; `p06` |
| the `(X, y)` bridge | Session 1 §6–7; `p05`, `p14`(a) |
| kNN mechanics by hand | Session 2 §2–3; `p04`, `p07` |
| choosing $k$, overfitting story | Session 2 §4; Session 3 §5; `p11`, `p15` |
| scaling distortion + standardization | Session 2 §5–6; `p02`, `p08`, `p12`, `p16`, `p18` |
| pipelines and leakage | Session 3 §2–3, §6; `p09`, `p17` |
| cross-validation mechanics | Session 3 §4; `p03`, `p10`, `p13` |
| honest protocol end-to-end | Session 3 §5, §7; `p14`, `p15`, `p19` |

## Quiz answers

<details><summary><b>Answers 1–15</b> (open only after committing to yours)</summary>

1. A Series (1-D) vs a one-column DataFrame (2-D); `.to_numpy()`
   gives `(90,)` vs `(90, 1)`.
2. `df[(df["mass_g"] > 0.7) & (df["species"] == "cava")]` — `&` binds
   tighter than `>`, so the original evaluates
   `0.7 & (...)` first: wrong computation (TypeError or garbage).
3. It keeps the row index out of the file (no `Unnamed: 0` column on
   reload).
   An `object` dtype on a numeric-looking column means at least one
   cell is not a number (typo, unit string, placeholder).
4. ```python
   FEATURES = ["temp_c", "vibration_mm_s"]
   X = df[FEATURES].to_numpy()
   y = df["status"].to_numpy()
   ```
5. Squared distances: $(0,0)\!: 2$; $(2,0)\!: 2$; $(0,3)\!: 5$;
   $(4,4)\!: 18$.
   $k=1$: tie at distance² 2 between two label-0 rows — either way
   label **0**.
   $k=3$: labels $\{0, 0, 1\}$ → **0**.
6. Each training row's nearest neighbor is itself (distance 0), so
   its own label wins: perfect training score with no generalization
   content — **overfitting** (memorization).
7. `(7, 50, 4)`; `(7, 50)`; `(7,)`.
8. $z = (14 - 10)/2 = 2$; the standardized column has mean 0 and
   variance 1 — subtracting a constant shifts the mean to 0 and
   leaves variance unchanged; dividing by $\sigma$ divides variance
   by $\sigma^2$ (F5 scaling rule).
9. $(100/0.1)^2 = 10^6$ — the big-spread feature outweighs the other
   a million-fold and effectively picks the neighbors alone.
10. (i) A single query/test row has no spread of its own to
    standardize by; (ii) letting test rows into $\mu, \sigma$ leaks
    evaluation data into the model's coordinates, so the test stops
    measuring generalization.
11. `fit`: scaler fits on `X_tr` (stores $\mu, \sigma$) → transforms
    `X_tr` → kNN fits on the result.
    `score`: `X_te` is *transformed with the stored training
    statistics* (never refit), then predicted and compared to
    `y_te`.
12. The clone-per-fold refits the scaler on each fold's training
    portion only, so validation rows never touch the statistics.
    Pre-scaled protocol: every $\mu_j, \sigma_j$ was computed from
    all rows — including each fold's validation rows — before the
    split.
13. 5 models, each trained on 32 rows; each row validated exactly
    once.
    Mean of fold accuracies = pooled accuracy exactly when all folds
    have equal size (weighted vs unweighted average coincide).
14. test > mean CV > best fold.
    The final test number measured a pre-chosen model on rows that
    influenced nothing; the mean CV of the chosen model is honest
    data-flow but selection-tilted (the winner was picked for it);
    the best fold is the max of noisy draws — upward-biased by
    construction.
15. k-nearest neighbors only (paraphrased rule: sklearn allowed,
    model family restricted to kNN); the toolkit:
    `KNeighborsClassifier`, `StandardScaler`, `Pipeline`,
    `train_test_split`, `cross_val_score` (plus dataset loaders).

</details>